# How To: Merge Fragmented Detections

When a single colony is split into multiple objects (fragmentation),
use refiners to merge the fragments back together.

In [1]:
from phenotypic.data import load_yeast_plate
from phenotypic.enhance import GaussianBlur, CLAHE
from phenotypic.detect import OtsuDetector
from phenotypic.refine import NearestNeighborMerger, MaskFill, MaskCloser

In [2]:
plate = load_yeast_plate()
plate = GaussianBlur(sigma=2.0).apply(plate)
plate = CLAHE(clip_limit=0.01).apply(plate)
plate = OtsuDetector().apply(plate)
print(f"Before merging: {plate.num_objects} objects")
plate.dash(overlay=True)

Before merging: 9 objects


## Morphological Closing

`MaskCloser` bridges small gaps between nearby fragments using dilation
followed by erosion.

In [3]:
closed = MaskCloser(width=5).apply(plate.copy())
closed = MaskFill().apply(closed)
print(f"After closing + fill: {closed.num_objects} objects")
closed.dash(overlay=True)

After closing + fill: 8 objects


## Nearest-Neighbor Merging

`NearestNeighborMerger` merges objects that are within a specified
distance of each other, assigning smaller fragments to their nearest
larger neighbor.

In [4]:
merged = NearestNeighborMerger().apply(plate.copy())
print(f"After NN merge: {merged.num_objects} objects")
merged.dash(overlay=True)

After NN merge: 9 objects


Use `MaskCloser` for small gaps between fragments, and
`NearestNeighborMerger` when fragments are further apart but clearly
belong to the same colony.